## Step 1: Mount Google Drive and Unzip Dataset

We mount Google Drive and unzip the `7thmaythesis.zip` file, which contains:
- `Annotation1/` with TIFF and XML
- `Annotation2/` with TIFF and XML
- `annotation_1_and_2_final_reshaped_combined.xlsx` for villi class labels

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Unzip the dataset
!unzip "/content/drive/MyDrive/7thMayThesis.zip" -d "/content/data"

Archive:  /content/drive/MyDrive/7thMayThesis.zip
   creating: /content/data/Annotation1/
  inflating: /content/data/Annotation1/Annotation1.tif  
  inflating: /content/data/Annotation1/Annotation1.xml  
   creating: /content/data/Annotation2/
  inflating: /content/data/Annotation2/Annotation2.tif  
  inflating: /content/data/Annotation2/Annotation2.xml  
  inflating: /content/data/Annotation1And2Combined.xlsx  


In [8]:
!ls /content/data/

Annotation1  Annotation1And2Combined.xlsx  Annotation2


🧩 Step 2: Load Polygon Annotations and Labels
In this step, we’ll:

Load the Excel file that maps annotation IDs to villi types

Parse the XML files to extract polygon coordinates

Link each annotation to its class label

In [9]:
# 📚 Imports
import xml.etree.ElementTree as ET
import pandas as pd

# 📍 Paths
excel_path = "/content/data/Annotation1And2Combined.xlsx"
xml_paths = {
    "Annotation1": "/content/data/Annotation1/Annotation1.xml",
    "Annotation2": "/content/data/Annotation2/Annotation2.xml"
}

# 📑 Load the Excel file into a dictionary: { "Annotation 10": "Terminal villi", ... }
label_df = pd.read_excel(excel_path)
label_map = dict(zip(label_df["annotation_id"], label_df["class"]))

# 🔍 Parse XML annotations from one file
def parse_xml_annotations(xml_path, source_name):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []

    for annotation in root.findall(".//annotation"):
        ann_id = annotation.attrib.get("name")
        points = [(int(p.attrib["x"]), int(p.attrib["y"])) for p in annotation.findall("p")]
        cls = label_map.get(ann_id)
        if cls is None:
            continue  # skip if no label
        annotations.append({
            "id": ann_id,
            "class": cls,
            "polygon": points,
            "source": source_name
        })

    return annotations

# 🔄 Parse both XML files
all_annotations = []
for source, path in xml_paths.items():
    all_annotations.extend(parse_xml_annotations(path, source))

# ✅ Done
print(f"✅ Loaded {len(all_annotations)} labeled polygons across both sources.")
print(f"🧪 Example: {all_annotations[0]}")

✅ Loaded 1187 labeled polygons across both sources.
🧪 Example: {'id': 'Annotation 305', 'class': 'Terminal villi', 'polygon': [(994, 2969), (915, 2991), (915, 2991), (915, 2992), (915, 2997), (914, 3002), (913, 3007), (913, 3012), (912, 3016), (912, 3021), (912, 3027), (912, 3033), (913, 3038), (915, 3042), (918, 3047), (920, 3051), (923, 3056), (926, 3060), (929, 3064), (931, 3068), (934, 3073), (938, 3078), (941, 3081), (945, 3085), (949, 3087), (954, 3090), (958, 3093), (962, 3095), (967, 3097), (973, 3098), (978, 3099), (983, 3099), (988, 3099), (994, 3099), (999, 3097), (1005, 3095), (1009, 3093), (1014, 3092), (1019, 3090), (1023, 3088), (1028, 3087), (1033, 3085), (1037, 3081), (1042, 3078), (1046, 3075), (1050, 3072), (1054, 3067), (1057, 3063), (1059, 3058), (1061, 3053), (1061, 3048), (1062, 3043), (1063, 3039), (1063, 3034), (1063, 3028), (1063, 3023), (1062, 3018), (1061, 3014), (1059, 3009), (1057, 3004), (1055, 2999), (1053, 2994), (1051, 2990), (1049, 2985), (1046, 2981)

📦 Step 3: Convert Polygons to Bounding Boxes
Now we’ll convert each polygon (list of x/y points) into a bounding box in (x_min, y_min, x_max, y_max) format, which is required for YOLO or COCO training.

In [10]:
# 📦 Convert polygon to bounding box
def polygon_to_bbox(points):
    xs, ys = zip(*points)
    return min(xs), min(ys), max(xs), max(ys)

# 🧮 Add bounding boxes to each annotation
for ann in all_annotations:
    ann["bbox"] = polygon_to_bbox(ann["polygon"])

# ✅ Preview one
print("Example with bounding box:")
print(all_annotations[0])

Example with bounding box:
{'id': 'Annotation 305', 'class': 'Terminal villi', 'polygon': [(994, 2969), (915, 2991), (915, 2991), (915, 2992), (915, 2997), (914, 3002), (913, 3007), (913, 3012), (912, 3016), (912, 3021), (912, 3027), (912, 3033), (913, 3038), (915, 3042), (918, 3047), (920, 3051), (923, 3056), (926, 3060), (929, 3064), (931, 3068), (934, 3073), (938, 3078), (941, 3081), (945, 3085), (949, 3087), (954, 3090), (958, 3093), (962, 3095), (967, 3097), (973, 3098), (978, 3099), (983, 3099), (988, 3099), (994, 3099), (999, 3097), (1005, 3095), (1009, 3093), (1014, 3092), (1019, 3090), (1023, 3088), (1028, 3087), (1033, 3085), (1037, 3081), (1042, 3078), (1046, 3075), (1050, 3072), (1054, 3067), (1057, 3063), (1059, 3058), (1061, 3053), (1061, 3048), (1062, 3043), (1063, 3039), (1063, 3034), (1063, 3028), (1063, 3023), (1062, 3018), (1061, 3014), (1059, 3009), (1057, 3004), (1055, 2999), (1053, 2994), (1051, 2990), (1049, 2985), (1046, 2981), (1042, 2977), (1038, 2974), (1034,

🧩 Step 4: Tile the Large TIFF Images into 1024×1024 Patches with 25% Overlap
We’ll tile the original Annotation1.tif and Annotation2.tif into 1024×1024 tiles with 25% (256px) overlap. This ensures that large villi spanning tile edges are captured more reliably. We’ll also track the top-left corner of each tile to adjust bounding box coordinates later.

In [20]:
from PIL import Image
import os

# Paths to TIFFs
tiff_paths = {
    "Annotation1": "/content/data/Annotation1/Annotation1.tif",
    "Annotation2": "/content/data/Annotation2/Annotation2.tif"
}

# Output folder for tiles
output_dir = "/content/tiles_1024"
os.makedirs(output_dir, exist_ok=True)

# Tile size and overlap settings
tile_size = 1024
overlap = 0.25  # 25%
stride = int(tile_size * (1 - overlap))  # = 768

# Store tile metadata
tile_info = {}
tile_counter = 0

for source, tiff_path in tiff_paths.items():
    img = Image.open(tiff_path)
    width, height = img.size

    for top in range(0, height, stride):
        if top + tile_size > height:
            top = height - tile_size  # ensure full tile
        for left in range(0, width, stride):
            if left + tile_size > width:
                left = width - tile_size

            right = left + tile_size
            bottom = top + tile_size

            tile = img.crop((left, top, right, bottom))
            tile_id = f"{source}_tile_{tile_counter:05d}"
            tile_path = os.path.join(output_dir, f"{tile_id}.png")
            tile.save(tile_path)

            tile_info[tile_id] = {
                "source": source,
                "x_offset": left,
                "y_offset": top,
                "width": tile_size,
                "height": tile_size,
                "path": tile_path
            }

            tile_counter += 1

print(f"✅ Tiled {tile_counter} overlapping tiles into {output_dir}")
print(f"🧱 Example tile info:\n{list(tile_info.items())[0]}")


✅ Tiled 382 overlapping tiles into /content/tiles_1024
🧱 Example tile info:
('Annotation1_tile_00000', {'source': 'Annotation1', 'x_offset': 0, 'y_offset': 0, 'width': 1024, 'height': 1024, 'path': '/content/tiles_1024/Annotation1_tile_00000.png'})


🧩 Step 5: Assign Bounding Boxes to Overlapping Tiles
Now that the TIFFs are tiled with 25% overlap, we’ll:

Check which global bounding boxes intersect with each tile

Adjust their coordinates to tile-local reference

Keep any bounding boxes that partially or fully fall within the tile

This ensures all villi — even those on the tile edges — are captured.

In [21]:
# 🧠 Utility: Check if two boxes intersect
def boxes_intersect(box1, box2):
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    return not (x1_max <= x2_min or x1_min >= x2_max or y1_max <= y2_min or y1_min >= y2_max)

# ✏️ Adjust global bbox to tile-local
def adjust_bbox_to_tile(bbox, tile_offset):
    x_min, y_min, x_max, y_max = bbox
    offset_x, offset_y = tile_offset
    return (
        max(0, x_min - offset_x),
        max(0, y_min - offset_y),
        min(tile_size, x_max - offset_x),
        min(tile_size, y_max - offset_y)
    )

# 🧩 Assign bounding boxes to tiles
tile_annotations = {tile_id: [] for tile_id in tile_info}

for ann in all_annotations:
    global_bbox = ann["bbox"]

    for tile_id, tile in tile_info.items():
        if ann["source"] != tile["source"]:
            continue  # skip other source

        tile_box = (
            tile["x_offset"],
            tile["y_offset"],
            tile["x_offset"] + tile["width"],
            tile["y_offset"] + tile["height"]
        )

        if boxes_intersect(global_bbox, tile_box):
            adjusted_bbox = adjust_bbox_to_tile(global_bbox, (tile["x_offset"], tile["y_offset"]))
            tile_annotations[tile_id].append({
                "class": ann["class"],
                "bbox": adjusted_bbox
            })

# ✅ Example output
example_tile = list(tile_annotations.keys())[0]
print(f"🧩 Tile {example_tile} has {len(tile_annotations[example_tile])} boxes.")
print(tile_annotations[example_tile][:2])  # preview first two boxes

🧩 Tile Annotation1_tile_00000 has 7 boxes.
[{'class': 'Terminal villi', 'bbox': (866, 476, 1024, 602)}, {'class': 'Terminal villi', 'bbox': (1010, 709, 1024, 856)}]


🧩 Step 6: Export Dataset in YOLOv8 Format (with Overlapping Tiles)
Now that each tile has its own set of bounding boxes (adjusted for overlap), we’ll:

Map class names to class IDs (for YOLO)

Normalize bounding box coordinates

Save them in .txt files with the same base name as the tile image

Each line in a label file will follow the format:
class_id x_center_norm y_center_norm width_norm height_norm

In [22]:
import cv2
import os

# 📂 Output directory for YOLO labels
labels_dir = "/content/yolo_labels"
os.makedirs(labels_dir, exist_ok=True)

# 🧭 Class name to ID mapping
class_names = sorted(set(ann["class"] for ann in all_annotations))
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("📚 Class to ID mapping:", class_to_id)

# 📝 Write YOLO label files
for tile_id, ann_list in tile_annotations.items():
    tile_path = tile_info[tile_id]["path"]
    img = cv2.imread(tile_path)
    h, w = img.shape[:2]

    label_lines = []

    for ann in ann_list:
        x_min, y_min, x_max, y_max = ann["bbox"]
        class_id = class_to_id[ann["class"]]

        # Normalize to (0–1) range
        x_center = (x_min + x_max) / 2 / w
        y_center = (y_min + y_max) / 2 / h
        bbox_width = (x_max - x_min) / w
        bbox_height = (y_max - y_min) / h

        label_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}")

    # Save only if there are annotations
    if label_lines:
        label_filename = f"{os.path.splitext(os.path.basename(tile_path))[0]}.txt"
        label_path = os.path.join(labels_dir, label_filename)
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))

📚 Class to ID mapping: {'Intermediate villi': 0, 'Stem villi': 1, 'Terminal villi': 2}


🧩 Step 7: Prepare YOLOv8 Dataset with 80/10/10 Train/Val/Test Split
We’ll now:

Create train/, val/, and test/ splits

Copy each image and its .txt file (only if labels exist)

Create the required data.yaml (used for training)

Leave test/ for manual evaluation later (YOLO doesn't use it during training)

In [25]:
import shutil
import random
from pathlib import Path

# 🗂 Paths
tiles_path = Path("/content/tiles_1024")
labels_path = Path("/content/yolo_labels")
dataset_root = Path("/content/yolo_dataset")

# Output dirs
splits = ["train", "val", "test"]
for split in splits:
    (dataset_root / "images" / split).mkdir(parents=True, exist_ok=True)
    (dataset_root / "labels" / split).mkdir(parents=True, exist_ok=True)

# 🔀 80/10/10 split (only keep images with labels)
all_tile_images = [img for img in tiles_path.glob("*.png") if (labels_path / f"{img.stem}.txt").exists()]
random.seed(42)
random.shuffle(all_tile_images)
n = len(all_tile_images)
train_files = all_tile_images[:int(0.8 * n)]
val_files = all_tile_images[int(0.8 * n):int(0.9 * n)]
test_files = all_tile_images[int(0.9 * n):]

# 🚚 Copy function
def copy_image_and_label(image_paths, split):
    for img_path in image_paths:
        label_path = labels_path / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy(img_path, dataset_root / "images" / split / img_path.name)
            shutil.copy(label_path, dataset_root / "labels" / split / label_path.name)

copy_image_and_label(train_files, "train")
copy_image_and_label(val_files, "val")
copy_image_and_label(test_files, "test")

# 📝 Create data.yaml for training
yaml_path = dataset_root / "data.yaml"
with open(yaml_path, "w") as f:
    f.write(f"""
path: {dataset_root}
train: images/train
val: images/val

names: {list(class_to_id.keys())}
""")

print(f"✅ YOLOv8 dataset ready with {len(train_files)} train, {len(val_files)} val, {len(test_files)} test tiles.")
print(f"📄 data.yaml path: {yaml_path}")

✅ YOLOv8 dataset ready with 305 train, 38 val, 39 test tiles.
📄 data.yaml path: /content/yolo_dataset/data.yaml


🧩 Step 7.5: Oversample Stem Villi Tiles Before Training
Since Stem villi are underrepresented in the dataset, we’ll duplicate tiles that contain them to boost their presence during training.

This improves the model’s ability to detect rare classes without modifying any YOLOv8 code.

In [28]:
import shutil
from pathlib import Path

# 📂 Training directories
label_train_dir = Path("/content/yolo_dataset/labels/train")
image_train_dir = Path("/content/yolo_dataset/images/train")

# 🧭 Class ID for 'Stem villi'
stem_class_id = class_to_id["Stem villi"]

# 🔁 Oversampling factor (how many times to duplicate each tile)
oversample_factor = 3  # Try 3x or 5x

# 🔍 Find all tiles that contain Stem villi
stem_tile_basenames = []

for label_file in label_train_dir.glob("*.txt"):
    with open(label_file) as f:
        if any(line.startswith(str(stem_class_id)) for line in f):
            stem_tile_basenames.append(label_file.stem)

print(f"🔎 Found {len(stem_tile_basenames)} tiles with Stem villi.")

# 🚚 Duplicate those tiles
for stem_tile in stem_tile_basenames:
    img_path = image_train_dir / f"{stem_tile}.png"
    lbl_path = label_train_dir / f"{stem_tile}.txt"

    for i in range(oversample_factor):
        new_img = image_train_dir / f"{stem_tile}_stem_{i}.png"
        new_lbl = label_train_dir / f"{stem_tile}_stem_{i}.txt"
        shutil.copy(img_path, new_img)
        shutil.copy(lbl_path, new_lbl)

print(f"✅ Duplicated {len(stem_tile_basenames)} tiles x{oversample_factor} times.")

🔎 Found 140 tiles with Stem villi.
✅ Duplicated 140 tiles x3 times.


🚀 Final Step: Launch YOLOv8 Training (with More Epochs + Augmentations)
We’ll now:

Use YOLOv8s

Train for 200 epochs

Enable stronger augmentations like scaling, translation, shearing, and flipping

Improve robustness and generalization, especially for medical object variance

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO

# 🚀 Load the small but powerful model
model = YOLO("yolov8s.yaml")

# 🏋️ Train with extended epochs and aggressive augmentation
model.train(
    data="/content/yolo_dataset/data.yaml",
    epochs=200,           # extended for full convergence
    imgsz=1024,
    batch=8,              # tune this based on your GPU
    workers=2,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2
)

Ultralytics 8.3.128 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, p

train: Scanning /content/yolo_dataset/labels/train.cache... 725 images, 0 backgrounds, 0 corrupt: 100%|██████████| 725/725 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 916.9±38.8 MB/s, size: 1195.0 KB)


val: Scanning /content/yolo_dataset/labels/val.cache... 38 images, 0 backgrounds, 0 corrupt: 100%|██████████| 38/38 [00:00<?, ?it/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     40/200      6.21G      1.231      1.579      1.843         60       1024: 100%|██████████| 91/91 [00:28<00:00,  3.14it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:00<00:00, 10.98it/s]

                   all         38        410      0.564      0.514      0.548      0.354



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     41/200      6.21G      1.207      1.548      1.814        185       1024:  55%|█████▍    | 50/91 [00:15<00:13,  3.01it/s]